# Linear Probe — Fetal Planes Classification

Freeze the pre-trained encoder, attach a linear head, train on **fetal_planes_db**  
(composite `Plane + Brain_plane` labels). Evaluates accuracy and balanced accuracy each epoch.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
DATASET_ROOT            = "/path/to/fetal_planes_db"    # <-- set this
PATH_ENCODER_CHECKPOINT = "/path/to/mae_final.pt"       # <-- set this
CHECKPOINT_DIR          = "/path/to/checkpoints/linear_probe"

# ── Encoder ───────────────────────────────────────────────────────────────
ENCODER_TYPE  = "mae"    # mae | vicreg | timm
TIMM_MODEL    = ""       # only for ENCODER_TYPE=timm

# ── Training ──────────────────────────────────────────────────────────────
MAX_IMAGE_HEIGHT = 224
EPOCHS           = 100
BATCH_SIZE       = 64
LR               = 1e-3
WEIGHT_DECAY     = 0.0
NUM_WORKERS      = 4
SAVE_EVERY       = 10

In [ ]:
import sys
from pathlib import Path

import torch

PROJECT_ROOT = str(Path.cwd().resolve().parents[1])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from baselines.mae_classify.train import (
    load_encoder,
    train_linear_probe,
    load_linear_probe_from_checkpoint,
    make_fetal_planes_probe_dataloader,
    gather_predictions,
    confusion_matrix,
    plot_linear_probe_history,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1. Load encoder

In [ ]:
encoder, embed_dim = load_encoder(ENCODER_TYPE, PATH_ENCODER_CHECKPOINT, TIMM_MODEL, DEVICE)
print(f"embed_dim={embed_dim}")

## 2. Train

In [ ]:
history = train_linear_probe(
    encoder=encoder,
    embed_dim=embed_dim,
    dataset_root=DATASET_ROOT,
    max_image_height=MAX_IMAGE_HEIGHT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    num_workers=NUM_WORKERS,
    device=DEVICE,
    checkpoint_dir=CHECKPOINT_DIR,
    save_every=SAVE_EVERY,
    show_plot=True,
)
print(f"Best val accuracy:          {100 * max(history['val_acc']):.2f}%")
print(f"Best val balanced accuracy: {100 * max(history['val_bal_acc']):.2f}%")

## 3. Confusion matrix

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

PROBE_CKPT = os.path.join(CHECKPOINT_DIR, f"linear_probe_epoch_{SAVE_EVERY}.pt")

encoder_cm, embed_dim_cm = load_encoder(ENCODER_TYPE, PATH_ENCODER_CHECKPOINT, TIMM_MODEL, DEVICE)
probe_model, class_to_idx = load_linear_probe_from_checkpoint(PROBE_CKPT, encoder_cm, embed_dim_cm, DEVICE)

num_classes = len(class_to_idx)
idx_to_class = {i: name for name, i in sorted(class_to_idx.items(), key=lambda x: x[1])}

val_loader = make_fetal_planes_probe_dataloader(
    class_to_idx, DATASET_ROOT,
    max_image_height=MAX_IMAGE_HEIGHT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    split="val",
)

y_true, y_pred = gather_predictions(probe_model, val_loader, DEVICE)
cm = confusion_matrix(y_true, y_pred, num_classes)
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(14, max(5, num_classes * 0.35)))
for ax, mat, title in zip(axes, (cm_norm, cm), ("Normalized", "Counts")):
    im = ax.imshow(mat, interpolation="nearest", cmap="Blues")
    ticks = np.arange(num_classes)
    short = lambda s: (s[:22] + "…") if len(s) > 23 else s
    ax.set_xticks(ticks); ax.set_xticklabels([short(idx_to_class[i]) for i in range(num_classes)], rotation=55, ha="right")
    ax.set_yticks(ticks); ax.set_yticklabels([short(idx_to_class[i]) for i in range(num_classes)])
    ax.set(xlabel="Predicted", ylabel="True", title=title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle("Linear probe — confusion matrix (val)", y=1.02)
plt.tight_layout()
plt.show()